# 01 - Limpieza AEMET

Carga, limpieza, mapeo provincia→CCAA y agregación diaria por CCAA del histórico meteorológico de AEMET (2021-2025).

## 1. Carga de datos

In [1]:
import pandas as pd
import numpy as np
import pyarrow

aemet = pd.read_csv(
    "../data/raw/aemet/aemet_todas_estaciones_2021_2025.csv",
    parse_dates=["fecha"]
)

aemet.head()

,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,...,horaHrMax,hrMin,horaHrMin,pintMax,horaPIntMax,presMax,horaPresMax,presMin,horaPresMin,sol
0,2021-01-01,B013X,"ESCORCA, LLUC",ILLES BALEARS,490,"6,2","36,8","3,8",21:10,"8,7",...,15:50,68.0,00:00,"9,6",17:45,NaN,NaN,NaN,NaN,NaN
1,2021-01-02,B013X,"ESCORCA, LLUC",ILLES BALEARS,490,"2,8","5,9","-0,2",22:00,"5,7",...,Varias,71.0,03:00,"2,4",10:15,NaN,NaN,NaN,NaN,NaN
2,2021-01-03,B013X,"ESCORCA, LLUC",ILLES BALEARS,490,"3,0","2,4","0,0",00:10,"6,1",...,Varias,76.0,13:50,"6,0",02:25,NaN,NaN,NaN,NaN,NaN
3,2021-01-04,B013X,"ESCORCA, LLUC",ILLES BALEARS,490,"4,3","11,7","-0,4",22:00,"9,0",...,Varias,75.0,12:40,"7,8",Varias,NaN,NaN,NaN,NaN,NaN
4,2021-01-05,B013X,"ESCORCA, LLUC",ILLES BALEARS,490,"3,0","0,1","-0,3",02:30,"6,2",...,Varias,74.0,13:10,"9,0",04:35,NaN,NaN,NaN,NaN,NaN


In [2]:
aemet.info()

print("Filas:", aemet.shape[0])
print("Columnas:", aemet.shape[1])

print(aemet.dtypes)

print("\nValores nulos:")
print(aemet.isna().sum())

print("\nDuplicados:", aemet.duplicated().sum())

<class 'pandas.DataFrame'>
RangeIndex: 1549716 entries, 0 to 1549715
Data columns (total 27 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   fecha        1549716 non-null  datetime64[us]
 1   indicativo   1549716 non-null  str           
 2   nombre       1549716 non-null  str           
 3   provincia    1549716 non-null  str           
 4   altitud      1549716 non-null  int64         
 5   tmed         1517707 non-null  str           
 6   prec         1504546 non-null  str           
 7   tmin         1518188 non-null  str           
 8   horatmin     1507970 non-null  str           
 9   tmax         1518738 non-null  str           
 10  horatmax     1508972 non-null  str           
 11  dir          1277731 non-null  float64       
 12  velmedia     1281774 non-null  str           
 13  racha        1277731 non-null  str           
 14  horaracha    1277659 non-null  str           
 15  hrMedia      1495216 non-n

## 2. Selección de columnas

In [3]:
columnas = [
    "fecha",
    "indicativo",
    "nombre",
    "provincia",
    "altitud",
    "tmed",
    "tmin",
    "tmax",
    "prec",
    "velmedia",
    "racha",
    "hrMedia",
    "hrMax",
    "hrMin",
    "presMax",
    "presMin",
    "sol"
]

aemet = aemet[columnas].copy()

## 3. Normalización de texto (provincia, nombre)

In [4]:
aemet["provincia_norm"] = (
    aemet["provincia"]
    .str.upper()
    .str.strip()
)

aemet["nombre"] = aemet["nombre"].str.strip()

## 4. Mapeo provincia → CCAA

In [5]:
provincia_a_ccaa = {
    "ALMERIA": "Andalucia",
    "CADIZ": "Andalucia",
    "CORDOBA": "Andalucia",
    "GRANADA": "Andalucia",
    "HUELVA": "Andalucia",
    "JAEN": "Andalucia",
    "MALAGA": "Andalucia",
    "SEVILLA": "Andalucia",

    "HUESCA": "Aragon",
    "TERUEL": "Aragon",
    "ZARAGOZA": "Aragon",

    "ASTURIAS": "Asturias",

    "BALEARES": "Baleares",
    "ILLES BALEARS": "Baleares",

    "LAS PALMAS": "Canarias",
    "SANTA CRUZ DE TENERIFE": "Canarias",
    "STA. CRUZ DE TENERIFE": "Canarias",

    "CANTABRIA": "Cantabria",

    "AVILA": "Castilla y Leon",
    "BURGOS": "Castilla y Leon",
    "LEON": "Castilla y Leon",
    "PALENCIA": "Castilla y Leon",
    "SALAMANCA": "Castilla y Leon",
    "SEGOVIA": "Castilla y Leon",
    "SORIA": "Castilla y Leon",
    "VALLADOLID": "Castilla y Leon",
    "ZAMORA": "Castilla y Leon",

    "ALBACETE": "Castilla la Mancha",
    "CIUDAD REAL": "Castilla la Mancha",
    "CUENCA": "Castilla la Mancha",
    "GUADALAJARA": "Castilla la Mancha",
    "TOLEDO": "Castilla la Mancha",

    "BARCELONA": "Cataluna",
    "GIRONA": "Cataluna",
    "LLEIDA": "Cataluna",
    "TARRAGONA": "Cataluna",

    "ALICANTE": "Comunidad Valenciana",
    "CASTELLON": "Comunidad Valenciana",
    "VALENCIA": "Comunidad Valenciana",

    "BADAJOZ": "Extremadura",
    "CACERES": "Extremadura",

    "A CORUÑA": "Galicia",
    "LUGO": "Galicia",
    "OURENSE": "Galicia",
    "PONTEVEDRA": "Galicia",

    "MADRID": "Madrid",

    "MURCIA": "Murcia",

    "NAVARRA": "Navarra",

    "ARABA/ALAVA": "Pais Vasco",
    "BIZKAIA": "Pais Vasco",
    "GIPUZKOA": "Pais Vasco",

    "LA RIOJA": "La Rioja",

    "CEUTA": "Ceuta",
    "MELILLA": "Melilla"
}

aemet["ccaa"] = aemet["provincia_norm"].map(provincia_a_ccaa)

In [ ]:

sin_mapear = aemet[aemet["ccaa"].isna()]["provincia"].unique()
assert len(sin_mapear) == 0, f"Provincias sin mapear: {sin_mapear}"

## 5. Validación de fechas y duplicados

In [7]:
print("Fecha minima:", aemet["fecha"].min())
print("Fecha maxima:", aemet["fecha"].max())

print("Fechas nulas:", aemet["fecha"].isna().sum())

print(
    "Duplicados fecha+indicativo:",
    aemet.duplicated(subset=["fecha", "indicativo"]).sum()
)

# aemet = aemet.drop_duplicates(subset=["fecha", "indicativo"])

Fecha minima: 2021-01-01 00:00:00
Fecha maxima: 2025-12-31 00:00:00
Fechas nulas: 0
Duplicados fecha+indicativo: 0


## 6. Limpieza de variables numéricas

In [8]:
variables_numericas = [
    "altitud",
    "tmed",
    "tmin",
    "tmax",
    "prec",
    "velmedia",
    "racha",
    "hrMedia",
    "hrMax",
    "hrMin",
    "presMax",
    "presMin",
    "sol"
]

for col in variables_numericas:
    aemet[col] = (
        aemet[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .replace("nan", np.nan)
    )

    aemet[col] = pd.to_numeric(
        aemet[col],
        errors="coerce"
    )

aemet[variables_numericas].describe().T

,count,mean,std,min,25%,50%,75%,max
altitud,1549716.0,528.781255,449.991982,1.0,130.0,463.0,785.0,3092.0
tmed,1517707.0,15.838313,7.133331,-16.2,10.6,15.8,21.2,40.2
tmin,1518188.0,10.330037,6.833912,-25.4,5.4,10.5,15.5,50.0
tmax,1518738.0,21.348556,8.168700,-50.0,15.4,21.0,27.2,47.6
prec,1495421.0,1.753426,6.203222,0.0,0.0,0.0,0.2,710.8
velmedia,1281774.0,2.779535,1.865738,0.0,1.7,2.2,3.6,33.1
racha,1277731.0,9.613112,4.039087,0.0,6.7,8.9,11.7,65.6
hrMedia,1495216.0,65.016178,18.166007,1.0,52.0,67.0,79.0,100.0
hrMax,1480306.0,87.105404,13.291925,4.0,82.0,91.0,97.0,100.0
hrMin,1480559.0,47.183534,19.371149,1.0,32.0,47.0,61.0,100.0


## 7. Detección de valores anómalos

In [9]:
print("Temperatura minima > maxima:")
print((aemet["tmin"] > aemet["tmax"]).sum())

print("Humedad > 100:")
print((aemet["hrMedia"] > 100).sum())

print("Humedad < 0:")
print((aemet["hrMedia"] < 0).sum())

print("Precipitacion negativa:")
print((aemet["prec"] < 0).sum())

print("Velocidad del viento negativa:", (aemet["velmedia"] < 0).sum())
print("Racha negativa:", (aemet["racha"] < 0).sum())
print("Sol negativo:", (aemet["sol"] < 0).sum())
print("Presion maxima negativa:", (aemet["presMax"] < 0).sum())
print("Presion minima negativa:", (aemet["presMin"] < 0).sum())

Temperatura minima > maxima:
1
Humedad > 100:
0
Humedad < 0:
0
Precipitacion negativa:
0
Velocidad del viento negativa: 0
Racha negativa: 0
Sol negativo: 0
Presion maxima negativa: 0
Presion minima negativa: 0


In [10]:
aemet[aemet["tmin"] > aemet["tmax"]][
    ["fecha", "indicativo", "nombre", "provincia", "tmin", "tmax"]
]

,fecha,indicativo,nombre,provincia,tmin,tmax
1092540,2023-09-20,6084X,FUENGIROLA,MALAGA,50.0,-50.0


## 8. Corrección de anomalías

In [11]:
aemet = aemet.drop(
    aemet[
        (aemet["tmin"] == 50.0) &
        (aemet["tmax"] == -50.0)
    ].index
)

## 9. Cobertura por estación y por CCAA

In [12]:
cobertura_estaciones = (
    aemet.groupby("indicativo")
    .agg(
        fecha_inicio=("fecha", "min"),
        fecha_fin=("fecha", "max"),
        registros=("fecha", "count")
    )
    .reset_index()
)

cobertura_estaciones.head()

,indicativo,fecha_inicio,fecha_fin,registros
0,0002I,2021-04-23,2021-07-06,57
1,0009X,2021-01-01,2025-12-31,1776
2,0016A,2021-01-01,2025-12-31,1826
3,0016B,2021-01-01,2025-12-31,1795
4,0034X,2021-01-01,2025-12-31,1826


In [13]:
print(aemet.groupby("ccaa")["indicativo"].nunique())

ccaa
Andalucia               132
Aragon                   62
Asturias                 26
Baleares                 44
Canarias                 75
Cantabria                23
Castilla la Mancha       61
Castilla y Leon         104
Cataluna                 82
Ceuta                     1
Comunidad Valenciana     43
Extremadura              53
Galicia                  55
La Rioja                  8
Madrid                   23
Melilla                   1
Murcia                   28
Navarra                  22
Pais Vasco               35
Name: indicativo, dtype: int64


## 10. Guardado del dataset limpio (nivel estación)

In [14]:
aemet.to_csv(
    "../data/processed/aemet_limpio_2021_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Dataset limpio:", aemet.shape)
aemet.head()

Dataset limpio: (1549715, 19)


,fecha,indicativo,nombre,provincia,altitud,tmed,tmin,tmax,prec,velmedia,racha,hrMedia,hrMax,hrMin,presMax,presMin,sol,provincia_norm,ccaa
0,2021-01-01,B013X,"ESCORCA, LLUC",ILLES BALEARS,490,6.2,3.8,8.7,36.8,0.6,9.2,93.0,98.0,68.0,NaN,NaN,NaN,ILLES BALEARS,Baleares
1,2021-01-02,B013X,"ESCORCA, LLUC",ILLES BALEARS,490,2.8,-0.2,5.7,5.9,0.3,9.4,90.0,99.0,71.0,NaN,NaN,NaN,ILLES BALEARS,Baleares
2,2021-01-03,B013X,"ESCORCA, LLUC",ILLES BALEARS,490,3.0,0.0,6.1,2.4,0.0,5.8,94.0,99.0,76.0,NaN,NaN,NaN,ILLES BALEARS,Baleares
3,2021-01-04,B013X,"ESCORCA, LLUC",ILLES BALEARS,490,4.3,-0.4,9.0,11.7,0.8,5.6,91.0,99.0,75.0,NaN,NaN,NaN,ILLES BALEARS,Baleares
4,2021-01-05,B013X,"ESCORCA, LLUC",ILLES BALEARS,490,3.0,-0.3,6.2,0.1,0.8,8.9,85.0,99.0,74.0,NaN,NaN,NaN,ILLES BALEARS,Baleares


## 11. Análisis de nulos

In [15]:
nulos = (
    aemet.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

print(nulos)

sol               82.818518
presMin           74.490148
presMax           74.489374
racha             17.550646
velmedia          17.289760
hrMax              4.478824
hrMin              4.462498
hrMedia            3.516711
prec               3.503547
tmed               2.065477
tmin               2.034439
tmax               1.998948
nombre             0.000000
fecha              0.000000
indicativo         0.000000
altitud            0.000000
provincia          0.000000
provincia_norm     0.000000
ccaa               0.000000
dtype: float64


In [16]:
variables_meteo = [
    "tmed",
    "tmin",
    "tmax",
    "prec",
    "velmedia",
    "racha",
    "hrMedia",
    "hrMax",
    "hrMin",
    "presMax",
    "presMin",
    "sol"
]

print(
    aemet[variables_meteo]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

sol         82.818518
presMin     74.490148
presMax     74.489374
racha       17.550646
velmedia    17.289760
hrMax        4.478824
hrMin        4.462498
hrMedia      3.516711
prec         3.503547
tmed         2.065477
tmin         2.034439
tmax         1.998948
dtype: float64


In [17]:
nulos_estacion = (
    aemet.groupby("indicativo")[variables_meteo]
    .agg(lambda x: x.isna().mean() * 100)
)

nulos_estacion["nulos_media_pct"] = nulos_estacion.mean(axis=1)

nulos_estacion.sort_values("nulos_media_pct", ascending=False).head(20)

,tmed,tmin,tmax,prec,velmedia,racha,hrMedia,hrMax,hrMin,presMax,presMin,sol,nulos_media_pct
indicativo,,,,,,,,,,,,,
8501,100.000000,100.000000,100.000000,0.848010,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,91.737334
8084Y,98.716303,98.716303,98.716303,0.064185,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,91.351091
B691,50.358522,37.286266,38.389410,2.702703,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,77.394742
6332X,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,1.526718,1.526718,6.870229,75.826972
8058X,15.290520,15.290520,15.290520,6.422018,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,71.024465
3194U,88.525896,88.525896,88.525896,0.557769,58.565737,58.565737,87.250996,35.936255,35.936255,100.000000,100.000000,100.000000,70.199203
4193Y,3.918322,3.918322,3.918322,0.055188,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,67.650846
9262,2.136986,1.753425,1.863014,0.273973,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,67.168950
6100B,4.240052,4.240052,4.240052,0.913242,100.000000,100.000000,60.534899,100.000000,100.000000,100.000000,100.000000,100.000000,64.514025


In [18]:
resumen_variables = pd.DataFrame({
    "registros_validos": aemet[variables_meteo].notna().sum(),
    "registros_nulos": aemet[variables_meteo].isna().sum(),
    "porcentaje_nulos": aemet[variables_meteo].isna().mean() * 100
})

resumen_variables

,registros_validos,registros_nulos,porcentaje_nulos
tmed,1517706,32009,2.065477
tmin,1518187,31528,2.034439
tmax,1518737,30978,1.998948
prec,1495420,54295,3.503547
velmedia,1281773,267942,17.289760
racha,1277730,271985,17.550646
hrMedia,1495216,54499,3.516711
hrMax,1480306,69409,4.478824
hrMin,1480559,69156,4.462498
presMax,395342,1154373,74.489374


In [19]:
estaciones_por_variable = (
    aemet.groupby("indicativo")[variables_meteo]
    .apply(lambda x: x.notna().any())
    .sum()
    .sort_values()
)

estaciones_por_variable

sol         164
presMax     244
presMin     244
racha       741
velmedia    742
hrMin       858
hrMax       858
prec        866
hrMedia     867
tmin        876
tmax        876
tmed        876
dtype: int64

In [20]:
cobertura_ccaa = (
    aemet.groupby("ccaa")[variables_meteo]
    .apply(lambda x: x.notna().mean() * 100)
)

cobertura_ccaa.round(2)

,tmed,tmin,tmax,prec,velmedia,racha,hrMedia,hrMax,hrMin,presMax,presMin,sol
ccaa,,,,,,,,,,,,
Andalucia,96.96,96.96,96.96,94.70,84.61,84.28,96.04,94.77,94.76,32.67,32.67,10.92
Aragon,99.70,99.72,99.80,98.18,94.88,94.83,97.83,99.26,99.26,20.69,20.69,17.55
Asturias,99.03,99.03,99.03,97.01,94.88,94.21,98.78,98.57,98.57,31.56,31.55,25.21
Baleares,97.93,98.25,98.26,98.69,80.30,79.92,96.36,95.48,95.48,32.25,32.25,13.89
Canarias,97.82,97.83,97.87,97.98,96.73,96.44,96.72,97.80,97.83,24.48,24.48,19.65
Cantabria,97.74,97.74,97.74,92.98,97.69,97.36,98.03,94.11,94.11,34.13,34.13,15.58
Castilla la Mancha,94.51,94.51,94.51,95.26,69.63,69.19,92.08,88.68,88.68,28.94,28.94,17.68
Castilla y Leon,98.73,98.78,98.91,96.12,90.56,90.33,97.90,98.07,98.16,23.91,23.90,20.41
Cataluna,98.36,98.36,98.37,92.28,71.74,71.35,96.43,93.76,93.75,17.45,17.45,14.60


## 12. Agregación diaria por CCAA (Paso 1 de la agregación espacial)

Media diaria de cada variable meteorológica dentro de cada CCAA, sin ponderar todavía. Esto evita que el número de estaciones de una CCAA distorsione el peso que tendrá esa región en la agregación nacional (Paso 2), que se calculará en `03_integracion_datos` usando los pesos de `02_limpieza_ree`.

In [ ]:

variables_agregacion = [
    "tmed", "tmin", "tmax", "prec",
    "velmedia", "racha",
    "hrMedia", "presMax", "presMin",
    "sol"
]

media_ccaa_dia = (
    aemet.groupby(["fecha", "ccaa"])[variables_agregacion]
    .mean()
    .reset_index()
)

print("Filas:", media_ccaa_dia.shape[0])
print("CCAA unicas:", media_ccaa_dia["ccaa"].nunique())
print("Rango de fechas:", media_ccaa_dia["fecha"].min(), "-", media_ccaa_dia["fecha"].max())

Filas: 34694
CCAA unicas: 19
Rango de fechas: 2021-01-01 00:00:00 - 2025-12-31 00:00:00


In [ ]:

dias_totales = (aemet["fecha"].max() - aemet["fecha"].min()).days + 1

cobertura_ccaa_dias = (
    media_ccaa_dia.groupby("ccaa")["fecha"]
    .nunique()
    .sort_values()
)

print("Dias totales del periodo:", dias_totales)
print("\nDias cubiertos por CCAA:")
print(cobertura_ccaa_dias)

Dias totales del periodo: 1826

Dias cubiertos por CCAA:
ccaa
Andalucia               1826
Aragon                  1826
Asturias                1826
Baleares                1826
Canarias                1826
Cantabria               1826
Castilla la Mancha      1826
Castilla y Leon         1826
Cataluna                1826
Ceuta                   1826
Comunidad Valenciana    1826
Extremadura             1826
Galicia                 1826
La Rioja                1826
Madrid                  1826
Melilla                 1826
Murcia                  1826
Navarra                 1826
Pais Vasco              1826
Name: fecha, dtype: int64


In [ ]:

print("% de nulos tras agregar por CCAA:")
print(media_ccaa_dia[variables_agregacion].isna().mean().mul(100).round(2))

% de nulos tras agregar por CCAA:
tmed        0.02
tmin        0.02
tmax        0.02
prec        0.12
velmedia    0.00
racha       0.03
hrMedia     0.00
presMax     0.09
presMin     0.09
sol         0.03
dtype: float64


## 13. Guardado del agregado diario por CCAA

In [24]:
media_ccaa_dia.to_parquet(
    "../data/processed/aemet_media_ccaa_diaria.parquet",
    index=False
)

print("Guardado:", media_ccaa_dia.shape)
media_ccaa_dia.head()

Guardado: (34694, 12)


,fecha,ccaa,tmed,tmin,tmax,prec,velmedia,racha,hrMedia,presMax,presMin,sol
0,2021-01-01,Andalucia,7.754464,4.417857,11.099107,0.382727,4.078571,12.128866,67.778761,979.944737,976.839474,5.053846
1,2021-01-01,Aragon,2.174138,-1.320690,5.662069,0.242105,3.398214,10.560714,76.245614,935.409091,933.100000,5.436364
2,2021-01-01,Asturias,2.724000,0.808000,4.644000,12.655000,2.645455,11.914286,90.920000,968.388889,963.500000,0.671429
3,2021-01-01,Baleares,8.787179,5.285000,12.253846,15.497500,2.761290,13.426667,83.179487,1002.716667,995.783333,1.860000
4,2021-01-01,Canarias,14.648649,12.179730,17.133784,0.333784,3.595775,10.078571,66.891892,986.611111,983.850000,2.853333
